# Cheap + light derivations (companion to `radiomap_6g_sweep.ipynb`)

Fills the **inexpensive** entries of `derived.json` so the main notebook's orchestrator only has the heavy ones left. The main orchestrator skips anything already derived (`have()`), so results from here are picked up on resume.

**Landing on the legacy values.** `legacy_published.json` records what the old pipeline used and published (hidden 64, 3 layers, lr 0.01, kNN k=6, 300 epochs, patience 30, 15 % hold-out, IDW power 2, spherical kriging with 6 lags, k-means `n_init` 10, ...). Every search below treats that literal as the **incumbent**: it is kept unless a challenger beats it by more than 1 paired standard error on identical draws (`RM_INCUMBENT_SE`). So each entry either *lands on legacy* (the data cannot tell it from the best candidate) or *moves off it*, with the margin written into its `source` string. Nothing is forced; `landing_vs_legacy.csv` at the end lists which landed and which did not. Landing means "indistinguishable from the best at this many seeds", not "proved optimal". The selection code lives in the main notebook (cell 10 and the `derive_*` functions), so the main notebook's own orchestrator applies the same rule.

| Step | Parameters | Cost |
|---|---|---|
| A | `huber_sigma_db` (4 scenarios x 3 densities) | seconds (MAD of residuals) |
| B | `kmeans_n_init` | about a minute (KMeans) |
| C | `idw_power_*`, `idw_nn_*`, `krig_model_*`, `krig_nlags_*` for **every** (scenario, density) cell, widened grids, incumbent rule | minutes (LOO / K-fold kriging, no NN training) |
| D | `spatial_folds`, `spatial_buffer_m` (needs B and C first) | seconds |
| E | the missing `knn_k` searches | **the costly one**: GCN inner CV. With `RM_KNN_SEARCH=climb` (default) about 3-5 of the 7 k values per cell are fitted instead of all 7 |
| F | *(optional, off)* `val_scheme`, `val_frac`, GCN/GAT `patience`, `epochs_cap`, `grad_clip` | provisional only, see the note in F |

**What this notebook does not move.** The data and protocol settings (`max_points`, `nopath_policy`, noise floor, unit offset, density grid, the held-out pool) decide which points exist, so they fix the pools every derivation here rests on. Section 2b prints where they differ from legacy (read-only). Changing them means re-deriving everything downstream.

**Before running**
- Put this file next to `radiomap_6g_sweep.ipynb`, `legacy_published.json` and `derived.json` (no `derived.complete`).
- Needs the same environment as the main notebook (torch, torch_geometric, deepmimo, pykrige, ...) and internet access to `deepmimo.net` (scenario discovery and loading).
- Step 1 re-executes the main notebook's definition cells (all `%%report` cells and the final orchestrator trigger are skipped). It also runs scenario discovery and loads the scenarios, so it is not instant.
- `derived.json` is backed up first; every step saves as it goes and is safe to re-run.

In [1]:
import os, json, shutil, time, math, contextlib
from datetime import datetime

MAIN_NB = "radiomap_6g_sweep.ipynb"     # path to the main notebook
WORKDIR = "."                           # folder that holds derived.json
LEGACY_JSON = "legacy_published.json"   # what the old pipeline used / published; its literals are the incumbents
RUN_DEPENDENT = False                   # step F (provisional), see its markdown first
PRUNE_OUT_OF_SCOPE = False              # True: drop registry entries of scenarios that are not in the frozen scn_list (see step C)

os.chdir(WORKDIR)
os.environ["RM_DERIVED_ONLY"] = "0"        # the main notebook must stay in pass 1 while deriving
os.environ["RM_LEGACY_REF"] = LEGACY_JSON  # read by the main notebook's cell 10 when it is loaded below
# os.environ["RM_KNN_SEARCH"] = "full"     # default "climb"; "full" fits the whole k grid (about 2x the cost)
# os.environ["RM_INCUMBENT_SE"] = "1.0"    # how many paired SEs a challenger must win by to displace the legacy value
os.environ.setdefault("RM_WORKERS", "6")   # parallel forked workers (~1 GB each). Use 1 on Windows (no fork) or low RAM.

assert os.path.exists("derived.json"), "derived.json not found in WORKDIR"
if not os.path.exists(LEGACY_JSON): print(f"WARNING: {LEGACY_JSON} not found; incumbents fall back to the LEGACY dict in the main notebook")
assert not os.path.exists("derived.complete"), "derived.complete exists -> main notebook is in pass 2; delete it to derive"
bak = f"derived.backup.{datetime.now():%Y%m%d_%H%M%S}.json"
shutil.copy("derived.json", bak); print("backup ->", bak)

@contextlib.contextmanager
def timed(label):
    t = time.time(); print(f"--- {label} ..."); yield
    print(f"--- {label}: {time.time() - t:.0f}s")

backup -> derived.backup.20261005_122149.json


## 1. Load the main notebook's definitions (no sweeps, no orchestrator)

In [2]:
nbk = json.load(open(MAIN_NB))
cells = nbk["cells"]
last = next(i for i, c in enumerate(cells) if c["cell_type"] == "code" and "def run_derivation" in "".join(c["source"]))
SKIP = set()    # optionally add cell indices to skip, e.g. {18} to skip the catalog listing

with timed("loading definitions"):
    for i, c in enumerate(cells[: last + 1]):
        if c["cell_type"] != "code" or i in SKIP: continue
        src = "".join(c["source"])
        if src.lstrip().startswith("%%report"): continue
        if i == last:                                   # keep the defs, drop the trailing `if PASS == 1: run_derivation(...)`
            src = src[: src.index("\nif PASS == 1:\n    run_derivation(")]
        res = get_ipython().run_cell(src, store_history=False)
        res.raise_error()
assert PASS == 1, "main notebook thinks it is in pass 2"
print("loaded cells 0..%d; PASS=%d" % (last, PASS))

--- loading definitions ...
/home/praaneshnair/gitProjects/prism/.venv/bin/python3
2026-10-05 12:21:49.417083
kept 17 of 85 entries


/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


python                          : 3.12.12 (main, Nov 19 2025, 22:46:53) [Clang 21.1.4 ]
platform                        : Linux-7.0.13-arch1-2-x86_64-with-glibc2.43
executable                      : /home/praaneshnair/gitProjects/prism/.venv/bin/python3
torch                           : 2.14.0+cu130
torch_geometric                 : 2.8.0.post1
numpy                           : 2.2.6
scipy                           : 1.18.1
sklearn                         : 1.9.1
pandas                          : 3.0.5
matplotlib                      : 3.11.2
pykrige                         : 1.7.3
deepmimo                        : 4.0.5
optuna                          : 5.0.0
tabulate                        : 0.10.0
cuda_available                  : False
device_used_in_this_notebook    : cpu
Audited 11 packages in 37ms
[pass 1] derivation stage
[registry] loaded derived.json
NUM_SEEDS = 16  (source: assumed)
[legacy] LEGACY dict agrees with legacy_published.json
i1_2p5
i2_28b


['i1_2p5', 'i2_28b']

Full catalog: 201 scenarios
Download failed: 403 Client Error: Forbidden for url: https://deepmimo.net/api/download/secure?filename=i3_2p5.zip
Download failed: 403 Client Error: Forbidden for url: https://deepmimo.net/api/download/secure?filename=i2_28.zip
  candidate i1_2p4           <- indoor slug pattern i<k>_...
  candidate i1_2p5           <- tag environment=indoor
  candidate i2_28            <- probe slug: direct download succeeded
  candidate i2_28b           <- tag environment=indoor
  candidate i3_2p4           <- indoor slug pattern i<k>_...
  candidate i3_2p5           <- probe slug: direct download succeeded
  candidate i3_60            <- indoor slug pattern i<k>_...
Scenario "i1_2p4" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 0 (tx_idx 0) & RXset 1 (rx_idxs 5000)
Loading TXRX PAIR: TXset 0 (tx_idx 1) & RXset 1 (rx_idxs 5000)
Loading TXRX PAIR: TXset 0 (tx_idx 2) & RXset 1 (rx_idxs 5000)
Loading TXRX PAIR: TXset 0 (t

...

Generating channels: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 990.36it/s]


,scenario,layout,freq_ghz,n_points,extent_m,los_frac,rss_min,rss_max,rss_std,ok,indoor_scale,note
0,i1_2p4,i1,2.4,5000,4.0,1.0,-88.97,-78.43,1.58,True,True,
1,i1_2p5,i1,2.5,5000,4.0,1.0,-61.92,-51.49,1.69,True,True,
2,i2_28,i2,NaN,0,NaN,NaN,NaN,NaN,NaN,False,False,FileNotFoundError: Scenario folder not found: ...
3,i2_28b,i2,28.0,4834,7.0,0.0,-91.97,-73.05,3.69,True,True,
4,i3_2p4,i3,2.4,5000,5.5,1.0,-60.62,-47.96,1.57,True,True,
5,i3_2p5,i3,NaN,0,NaN,NaN,NaN,NaN,NaN,False,False,FileNotFoundError: Scenario folder not found: ...
6,i3_60,i3,60.0,5000,5.5,1.0,-87.99,-76.13,1.24,True,True,



Usable indoor scenarios: ['i2_28b', 'i1_2p5', 'i1_2p4', 'i3_2p4', 'i3_60']
Distinct layout families: 3 | distinct carrier bands (GHz): [np.float64(2.4), np.float64(2.5), np.float64(28.0), np.float64(60.0)]
Ordered train->test pairs by shift type: {'layout+band': 12, 'identical': 2, 'layout-only': 4, 'band-only': 2}

[RESULT] 3 distinct layouts available. Section 6f / 7c / 8c / 8d extend every sweep to them.
[resume] discovery also found ['i1_2p4']; ignored so derived.json stays consistent (delete scn_list from derived.json to include them)
Scenario "i2_28b" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 1 (tx_idx 0) & RXset 0 (rx_idxs 5000)


Generating channels: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 60.02it/s]


[load_scenario] i2_28b tx=0: dropped 166/5000 no-path / sub-noise-floor points
(4834, 3) (4834,) (4834, 9) 28.0
--- loading definitions: 25s
loaded cells 0..125; PASS=1


## 2. Rebuild the pools exactly as the orchestrator does

In [3]:
FROZEN = D("scn_list")
if list(SCN_LIST) != list(FROZEN):
    print("WARNING: discovery returned", SCN_LIST, "but the frozen scn_list is", FROZEN, "-> using the frozen one")
    SCN_LIST = list(FROZEN)

with timed("build_data"):
    build_data()
import hashlib
for scn in SCN_LIST:                        # same rule as the orchestrator's N5: register the pool on first build, verify on every later one
    fp = hashlib.md5(DATA[scn][4].tobytes()).hexdigest()
    if not have("pool_fingerprint", scn):
        set_derived("pool_fingerprint", fp, "convention: pool identity check", "N5", scn=scn)
    else:
        assert D("pool_fingerprint", scn) == fp, f"{scn}: pool differs from the saved derivation; delete derived.json and rerun"
save_derived()
verify_pool_identity()                      # pools must match the saved fingerprints, otherwise derived values would leak
print("pool identity OK for", SCN_LIST)

DENS_ALL = [d for d in _grid if d <= POOL_N // 3]
DENS_T = [DENS_ALL[0], DENS_ALL[len(DENS_ALL) // 2], DENS_ALL[-1]]     # tuning densities, as in run_derivation
DENS_D = DENS_T
print("DENS_T =", DENS_T, "| derivation seeds =", SEEDS_D, "| workers =", N_WORKERS)

def status():
    df = derivation_report()
    a = df[df.source.astype(str).str.startswith("assumed")]
    b = df[df.source.astype(str).str.contains("BOUNDARY")]
    print(f"entries={len(df)}  still assumed={len(a)} ({sorted(a.name.unique())})")
    print(f"BOUNDARY flags={len(b)}")
status()

--- build_data ...
Scenario "i2_28b" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 1 (tx_idx 0) & RXset 0 (rx_idxs 5000)


Generating channels: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 63.73it/s]


[load_scenario] i2_28b tx=0: dropped 166/5000 no-path / sub-noise-floor points
Scenario "i1_2p5" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 2 (tx_idx 0) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 1) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 2) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 3) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 4) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 5) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 6) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 7) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 8) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 9) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 10) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 11) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 12) & RXset 0 (rx_idxs 5000)
Loading

Generating channels: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 1463.69it/s]


Scenario "i3_2p4" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 2 (tx_idx 0) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 3 (tx_idx 0) & RXset 0 (rx_idxs 5000)


Generating channels: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 983.01it/s]


Scenario "i3_60" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 2 (tx_idx 0) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 3 (tx_idx 0) & RXset 0 (rx_idxs 5000)


Generating channels: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 976.27it/s]


--- build_data: 6s
pool identity OK for ['i2_28b', 'i1_2p5', 'i3_2p4', 'i3_60']
DENS_T = [20, 107, 327] | derivation seeds = (1, 2, 3, 4, 5) | workers = 6
entries=89  still assumed=57 (['band_log_tol', 'clip_test', 'clip_train', 'dropout', 'edge_weight_kind', 'epochs_cap', 'eps_dist', 'eps_edge', 'eps_idw', 'eps_improve', 'eps_norm', 'eps_rownorm', 'freq_exp', 'grad_clip', 'heads', 'hidden', 'huber_beta_db', 'huber_sigma_db', 'idw_nn_detrended', 'idw_nn_raw', 'idw_power_detrended', 'idw_power_raw', 'kernel_bw_m', 'kmeans_n_init', 'knn_k', 'krig_model_detrended', 'krig_model_raw', 'krig_nlags_detrended', 'krig_nlags_raw', 'krig_range', 'layers', 'lr', 'max_points', 'n_calib', 'noise_floor_dbm', 'nopath_policy', 'num_seeds', 'path_margin_db', 'patience', 'power_definition', 'rx_set_index', 'spatial_buffer_m', 'spatial_folds', 'star_K', 'trend_mode', 'tune_exclude_n', 'tx_index', 'unit_offset_db', 'val_frac', 'val_scheme', 'weight_decay'])
BOUNDARY flags=0


## 2b. Where the current registry differs from legacy (read-only)

These settings fix the pools, so they are **not** touched here. Items marked `False` explain why the final numbers can differ from `legacy_published.json` even when every hyper-parameter lands on its legacy value.

In [4]:
parity_df = parity_report()

Legacy protocol vs current registry: 17 of 30 items differ
                    item                       legacy                                current  same                                                                         note
               scenarios      [i1_2p5, i2_28b, i3_60]        [i2_28b, i1_2p5, i3_2p4, i3_60]  True same = every legacy scenario is present; extra scenarios here are additional
            density_grid [30, 50, 100, 200, 300, 500] [20, 35, 61, 107, 187, 327, 572, 1000] False          SPEC log-spaced grid vs the legacy list: table cells do not line up
      max_points[i2_28b]                         5000                                   5000  True                                           None = every point in the scenario
power_definition[i2_28b]                         mean                                   mean  True                                                                             
  unit_offset_db[i2_28b]                          0.0        

## A. `huber_sigma_db` (robust scale of detrended pool residuals)

In [5]:
with timed("huber_sigma_db"):
    derive_huber_sigma(); save_derived()
print({(k[1], k[2]): round(v["value"], 2) for k, v in DERIVED.items() if k[0] == "huber_sigma_db" and k[1]})

--- huber_sigma_db ...
--- huber_sigma_db: 0s
{('i2_28b', 20): 4.86, ('i2_28b', 107): 3.31, ('i2_28b', 327): 3.67, ('i1_2p5', 20): 1.54, ('i1_2p5', 107): 2.78, ('i1_2p5', 327): 2.79, ('i3_2p4', 20): 2.14, ('i3_2p4', 107): 2.02, ('i3_2p4', 327): 1.73, ('i3_60', 20): 1.64, ('i3_60', 107): 1.93, ('i3_60', 327): 1.8}


## B. `kmeans_n_init` (inertia check, 4 candidates x 5 seeds; legacy 10 kept if sufficient)

In [6]:
with timed("kmeans_n_init"):
    derive_kmeans_n_init(); save_derived()
print(DERIVED[("kmeans_n_init", None, None, None)]["source"])

--- kmeans_n_init ...
--- kmeans_n_init: 18s
derived: smallest-restart noise sd=0.15; sufficient n_init [10, 30]; legacy 10 kept (mean inertia {1: 28.4, 3: 28.2, 10: 28.0, 30: 27.9})


## C. IDW / kriging searches on widened grids, legacy as incumbent

The grids are supersets of the old ones (the old winners sat on their edges), and every grid contains the legacy value (IDW power 2 with all neighbours, spherical variogram with 6 lags). Because the selection rule changed (incumbent kept unless beaten by > 1 paired SE) and the kriging search now uses all 5 derivation seeds instead of 2, **every** (scenario, density) cell is re-run, not only the flagged ones. A `BOUNDARY` flag now only appears when a challenger that displaced legacy sits on the new grid edge; then widen the grid in cell 119 of the main notebook and re-run.

Entries for scenarios that are not in `SCN_LIST` (e.g. `i1_2p4`) are not re-run. Their old `BOUNDARY` flags would still make `final_checks()` refuse to finish, so they are listed here; set `PRUNE_OUT_OF_SCOPE = True` to drop them (`derived.json` was backed up first).

In [7]:
cells_c = [(s, n) for s in SCN_LIST for n in DENS_T]
print(f"re-running IDW + kriging for {len(cells_c)} (scenario, density) cells; grids: power {GRID_P}, nlags {GRID_NL}")
with timed("idw + kriging"):
    run_jobs([("derive_idw", c) for c in cells_c] + [("derive_krig", c) for c in cells_c])
    save_derived()

stale = sorted({k[1] for k in DERIVED if k[1] is not None and k[1] not in SCN_LIST})
if stale:
    n_stale = sum(1 for k in DERIVED if k[1] in stale); n_flag = sum(1 for k, v in DERIVED.items() if k[1] in stale and "BOUNDARY" in str(v["source"]))
    print(f"entries for out-of-scope scenarios {stale}: {n_stale} ({n_flag} flagged BOUNDARY, which would block final_checks())")
    if PRUNE_OUT_OF_SCOPE:
        for k in [k for k in DERIVED if k[1] in stale]: del DERIVED[k]
        save_derived(); print("pruned them from derived.json (backup kept)")
    else:
        print("kept; set PRUNE_OUT_OF_SCOPE = True and re-run this cell to drop them")

edge = [k for k, v in DERIVED.items() if "BOUNDARY" in str(v["source"]) and k[1] in SCN_LIST]
print("on the (new) edge after displacing legacy:", sorted(edge) or "none")
print("krig_range:", {s: round(D("krig_range", s), 3) for s in SCN_LIST})
off = [(k[0], k[1], k[2]) for k in DERIVED if k[0] in ("idw_power_raw", "idw_power_detrended", "krig_nlags_raw", "krig_nlags_detrended") and k[1] in SCN_LIST
       and DERIVED[k]["value"] != LEGACY[k[0]]]
print(f"idw_power / krig_nlags entries off legacy: {len(off)}")

re-running IDW + kriging for 12 (scenario, density) cells; grids: power (0.1, 0.25, 0.5, 1, 1.5, 2, 3, 4, 6, 8), nlags (3, 4, 6, 8, 12, 16, 24, 32)
--- idw + kriging ...
[parallel] 6 worker(s) for 24 job(s) (RM_WORKERS=6)


/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 7.487616472778308e-17.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 6.4515280697466895e-21.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.69667077380387e-20.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.1894672731711893e-20.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detecte

[parallel] job derive_idw('i1_2p5', 20) done (1/24)
[parallel] job derive_idw('i1_2p5', 107) done (2/24)
[parallel] job derive_idw('i1_2p5', 327) done (3/24)
[parallel] job derive_idw('i3_2p4', 20) done (4/24)
[parallel] job derive_idw('i3_2p4', 107) done (5/24)
[parallel] job derive_idw('i3_2p4', 327) done (6/24)
[parallel] job derive_idw('i3_60', 20) done (7/24)
[parallel] job derive_idw('i3_60', 107) done (8/24)
[parallel] job derive_idw('i3_60', 327) done (9/24)


/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.9669393042071128e-16.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 6.483980936969688e-17.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.1534554699628424e-16.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 7.881816976793424e-22.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detect

[parallel] job derive_krig('i1_2p5', 107) done (10/24)
[parallel] job derive_krig('i3_2p4', 107) done (11/24)


...

[parallel] job derive_krig('i1_2p5', 327) done (13/24)
[parallel] job derive_krig('i3_2p4', 20) done (14/24)
[parallel] job derive_krig('i3_2p4', 327) done (15/24)
[parallel] job derive_krig('i3_60', 107) done (16/24)
[parallel] job derive_idw('i2_28b', 20) done (17/24)
[parallel] job derive_idw('i2_28b', 107) done (18/24)
[parallel] job derive_idw('i2_28b', 327) done (19/24)
[parallel] job derive_krig('i3_60', 20) done (20/24)
[parallel] job derive_krig('i2_28b', 107) done (21/24)


/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.4029712906851352e-20.
  a_inv = scipy.linalg.inv(a)


[parallel] job derive_krig('i3_60', 327) done (22/24)


/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.1617244883282201e-19.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.706847594810098e-19.
  a_inv = scipy.linalg.inv(a)


[parallel] job derive_krig('i2_28b', 20) done (23/24)
[parallel] job derive_krig('i2_28b', 327) done (24/24)
--- idw + kriging: 80s
on the (new) edge after displacing legacy: [('idw_power_detrended', 'i2_28b', 20, None), ('idw_power_raw', 'i2_28b', 20, None), ('krig_nlags_detrended', 'i1_2p5', 107, None), ('krig_nlags_raw', 'i1_2p5', 20, None), ('krig_nlags_raw', 'i1_2p5', 107, None), ('krig_nlags_raw', 'i3_60', 327, None)]
krig_range: {'i2_28b': 1.166, 'i1_2p5': 3.832, 'i3_2p4': 1.897, 'i3_60': 3.391}
idw_power / krig_nlags entries off legacy: 29


## D. `spatial_folds` and `spatial_buffer_m` (uses B and the C-updated `krig_range`)

In [8]:
with timed("derive_spatial"):
    derive_spatial(); save_derived()
print("spatial_folds =", D("spatial_folds"), "|", DERIVED[("spatial_folds", None, None, None)]["source"][:110])
print("spatial_buffer_m:", {s: round(D("spatial_buffer_m", s), 2) for s in SCN_LIST})

--- derive_spatial ...
--- derive_spatial: 1s
spatial_folds = 2 | derived: largest fold count with every block >= 327 points and >= 327 candidates beyond the variogram-range bu
spatial_buffer_m: {'i2_28b': 1.17, 'i1_2p5': 3.83, 'i3_2p4': 1.9, 'i3_60': 3.39}


## E. Missing `knn_k` searches (inner-CV GCN, k in `KS`, 5 seeds x 5 folds)

This is the most expensive step in this notebook. It runs in parallel and saves after each finished job, so an interrupt keeps progress. As in the main orchestrator, it runs *before* GNN hyperparameter tuning, so it uses the current GNN defaults. The legacy k=6 is the incumbent: with `RM_KNN_SEARCH=climb` (default) it is fitted first, then the search walks to larger or smaller k only while the neighbour improves (assumes the CV score is roughly unimodal in k; use `full` to check). If a challenger that displaced k=6 is `BOUNDARY` (the smallest or largest k in the grid), widen `KS` and re-run just that cell.

In [9]:
import torch, heapq, time
torch.set_num_threads(1)

DENS_D = DENS_T
def knn_todo():
    """cells without a knn_k derived under the incumbent rule (an entry from the old rule has no 'legacy' in its source and is re-run)"""
    return [(s, n) for s in SCN_LIST for n in DENS_T if not (have("knn_k", s, n) and "legacy" in str(DERIVED[("knn_k", s, n, None)]["source"]))]

todo = knn_todo()
n_mid = DENS_T[1]

per_call = {}
for s in SCN_LIST:
    ei, ew = knn_graph(DATA[s][0], k=6)
    t = time.time()
    cv_gnn(GCN, s, n_mid, SEEDS_D[0], ei, ew)
    per_call[s] = time.time() - t
    print(f"{s}: {len(DATA[s][0]):>7} nodes, {per_call[s]:.1f}s per 5 fits")

jobs = []
for s, n in todo:
    grid_n = len([k for k in KS if k < n])
    n_calls = (min(grid_n, 4) if KNN_SEARCH == "climb" else grid_n) * len(SEEDS_D)   # one cv_gnn call = 5 fits; climb ~ incumbent + 2 neighbours + 1 step
    jobs.append(per_call[s] * n_calls)
jobs.sort(reverse=True)

workers = [0.0] * max(1, N_WORKERS)
heapq.heapify(workers)
for j in jobs:
    heapq.heappush(workers, heapq.heappop(workers) + j)

print(f"pending searches: {len(todo)} | KNN_SEARCH = {KNN_SEARCH}")
print(f"serial total ~ {sum(jobs)/3600:.2f} h | wall on {N_WORKERS} workers ~ {max(workers)/3600:.2f} h")

i2_28b:    4834 nodes, 17.9s per 5 fits
i1_2p5:    5000 nodes, 9.7s per 5 fits
i3_2p4:    5000 nodes, 19.8s per 5 fits
i3_60:    5000 nodes, 11.5s per 5 fits
pending searches: 12 | KNN_SEARCH = climb
serial total ~ 0.98 h | wall on 6 workers ~ 0.16 h


In [10]:
todo = knn_todo()
fits = sum((min(len([k for k in KS if k < n]), 4) if KNN_SEARCH == "climb" else len([k for k in KS if k < n])) * len(SEEDS_D) * 5 for s, n in todo)
print(f"{len(todo)} searches pending (~{fits} GCN fits, KNN_SEARCH={KNN_SEARCH}):", todo)
with timed("knn_k"):
    run_jobs([("derive_graph", c) for c in todo]); save_derived()
for s in SCN_LIST:
    print(s, {n: D("knn_k", s, n) for n in DENS_T if have("knn_k", s, n)})
print("BOUNDARY knn_k:", [k for k, v in DERIVED.items() if k[0] == "knn_k" and "BOUNDARY" in str(v["source"])] or "none")

12 searches pending (~1200 GCN fits, KNN_SEARCH=climb): [('i2_28b', 20), ('i2_28b', 107), ('i2_28b', 327), ('i1_2p5', 20), ('i1_2p5', 107), ('i1_2p5', 327), ('i3_2p4', 20), ('i3_2p4', 107), ('i3_2p4', 327), ('i3_60', 20), ('i3_60', 107), ('i3_60', 327)]
--- knn_k ...
[parallel] 6 worker(s) for 12 job(s) (RM_WORKERS=6)
[parallel] job derive_graph('i1_2p5', 107) done (1/12)
[parallel] job derive_graph('i3_2p4', 20) done (2/12)
[parallel] job derive_graph('i3_2p4', 327) done (3/12)
[parallel] job derive_graph('i1_2p5', 20) done (4/12)
[parallel] job derive_graph('i1_2p5', 327) done (5/12)
[parallel] job derive_graph('i3_2p4', 107) done (6/12)
[parallel] job derive_graph('i3_60', 20) done (7/12)
[parallel] job derive_graph('i2_28b', 20) done (8/12)
[parallel] job derive_graph('i2_28b', 107) done (9/12)
[parallel] job derive_graph('i3_60', 107) done (10/12)
[parallel] job derive_graph('i2_28b', 327) done (11/12)
[parallel] job derive_graph('i3_60', 327) done (12/12)
--- knn_k: 1380s
i2_28b 

## F. (optional, default off) provisional `val_scheme`, `val_frac`, `patience`, `epochs_cap`, `grad_clip`

These are cheap-ish, but in the main orchestrator they are derived **after** the GNN hyperparameter tuning, and `get_hp` feeds into them. Run now, they would see the untuned defaults, so they are written to `provisional_rules.json` only and **never** to `derived.json`. Use them as a preview of where the values land. Cost is about 250 validation-curve fits plus about 300 for the scheme comparison.

The same incumbent rule applies: the legacy 15 % hold-out stays unless beaten by > 1 paired SE; `patience` keeps the legacy 30 when it already meets the validation-curve criterion; `epochs_cap` starts its doubling search at the legacy 300; `grad_clip` stays at the legacy 5.0 when clipping is inactive.

In [11]:
def run_provisional():
    snap, real_save = dict(DERIVED), save_derived
    globals()["save_derived"] = lambda *a, **k: None            # nothing reaches derived.json
    try:
        derive_val_scheme(); derive_training_rules(("GCN", "GAT"))
        new = {k: v for k, v in DERIVED.items() if snap.get(k) is not v}
    finally:
        globals()["save_derived"] = real_save
        DERIVED.clear(); DERIVED.update(snap)
    json.dump({_k2s(k): v for k, v in new.items() if k[0] in ("val_scheme", "val_frac", "patience", "epochs_cap", "grad_clip")},
              open("provisional_rules.json", "w"), indent=1, default=str)
    print("wrote provisional_rules.json (", len(new), "entries ); derived.json untouched")

if RUN_DEPENDENT:
    with timed("provisional rules"): run_provisional()
else:
    print("skipped (RUN_DEPENDENT = False)")

skipped (RUN_DEPENDENT = False)


## Wrap-up

In [12]:
DENS_D = DENS_ALL                 # restore, as the orchestrator does
save_derived()
status()
landing_df = landing_table()      # derived value vs legacy literal for every experiment-derived entry -> landing_vs_legacy.csv

entries=217  still assumed=54 (['band_log_tol', 'clip_test', 'clip_train', 'dropout', 'edge_weight_kind', 'epochs_cap', 'eps_dist', 'eps_edge', 'eps_idw', 'eps_improve', 'eps_norm', 'eps_rownorm', 'freq_exp', 'grad_clip', 'heads', 'hidden', 'huber_beta_db', 'huber_sigma_db', 'idw_nn_detrended', 'idw_nn_raw', 'idw_power_detrended', 'idw_power_raw', 'kernel_bw_m', 'knn_k', 'krig_model_detrended', 'krig_model_raw', 'krig_nlags_detrended', 'krig_nlags_raw', 'krig_range', 'layers', 'lr', 'max_points', 'n_calib', 'noise_floor_dbm', 'nopath_policy', 'num_seeds', 'path_margin_db', 'patience', 'power_definition', 'rx_set_index', 'star_K', 'trend_mode', 'tune_exclude_n', 'tx_index', 'unit_offset_db', 'val_frac', 'val_scheme', 'weight_decay'])
BOUNDARY flags=6
Landing on legacy (derived entries whose value equals the legacy literal):
                name  entries  landed
      huber_sigma_db       12       0
    idw_nn_detrended       12       3
          idw_nn_raw       12       5
 idw_power_de